In [27]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import interrupt,Command
import requests

from langgraph.checkpoint.memory import MemorySaver

In [28]:
load_dotenv()

True

In [29]:
llm = ChatGoogleGenerativeAI(
model="gemini-2.5-flash",
# temperature=0.2,
# max_tokens=100,   
# timeout=None,
# max_retries=2,
# other params...
)


In [30]:
# *******
#    Tools 
# *******

@tool 
def get_stock_price(symbol:str) -> str:
    """ Fetch latest stock price for a given symbol (e.g., 'AAPL','TESLA')
    using Alpha Vantage API KEY in the URL."""

    url = (
            "https://www.alphavantage.co/query"
            f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey=7MP1XMG5WR2P61VE"
        )
    response = requests.get(url)
    return response.json()

In [31]:
@tool
def purchase_stock(symbol:str,quantity:int) -> dict:
    """ Simulate purchasing a given quantity of stock for a stock symbol.

    HUMAN-IIN-THE-LOOP:
    Before confirming the purchase,
    this tool will interrupt and wait for a human decision ("yes" / anything else)."""

    # This pauses the graph and returns control to the human user, waiting for their input.
    decision = interrupt(f"Appprove buying {quantity} shares of {symbol}? (yes/no)")

    if isinstance(decision,str) and decision.lower() == "yes":
        return {
            "status":"success",
            "message":f"Purchase order placed for {quantity} shares of {symbol}.",
            "quantity":quantity,
        }
    else:
        return {
            "status":"cancelled",
            "message":f"Purchase of {quantity} shares of {symbol} cancelled by user.",
            "quantity":0,
        }
    

In [32]:


tools = [get_stock_price,purchase_stock]
llm_with_tools = llm.bind_tools(tools)

In [33]:
# *******
#  State
# *******
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


In [42]:
# *********
#  Nodes
# *******

def chat_node(state:ChatState) -> ChatState:
    """LLM node that may answer or request a tool call."""

    messages = state["messages"]
    response = llm_with_tools.invoke(messages)

    return {"messages":[response]}

In [43]:
tool_node = ToolNode(tools)

# -------------------
# 5. Checkpointer (in-memory)
# -------------------
memory = MemorySaver()


In [44]:
# -------------------
# 5. Graph
# -------------------

graph = StateGraph(ChatState)
graph.add_node("chat_node",chat_node)
graph.add_node("tools",tool_node)

graph.add_edge(START,"chat_node")

graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools","chat_node")

chatbot = graph.compile(checkpointer=memory)



In [ ]:
# -------------------
# 7. Simple usage example (CLI with HITL)
# -------------------

# from pyexpat.errors import messages


if __name__ == "__main__":
    # use a fixed thread_id so the conversation is persisted in memory
    thread_id = "chatbot_hitl_thread"

    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in {"exit","quit"}:
            print("GoodBye!")
            break

        #  Built initial state for this turn
        state = {"messages":[HumanMessage(content=user_input)]}

        # Run the graph with the current state and thread_id for memory
        result = chatbot.invoke(
            state,
            config={"configurable":{"thread_id":thread_id}}
            
        )

        interrupts = result.get("__interrupt__",[])

        if interrupts:
            # Our interrupt paylaod is the string we passed to interrupt(...)
            prompt_to_human = interrupts[0].value
            print(f"HITL: {prompt_to_human}")
            decision = input("Your decision: ").strip().lower()

            #  Resume graph woth the human decision ("yes" / "no" / what ever)

            result = chatbot.invoke(
                Command(resume=decision),
                config={"configurable":{"thread_id":thread_id}}
            )
        
        # Get the latest message from the assistant
        messages = result["messages"]

        last_msg = messages[-1]
        print(f"{last_msg.content}")


[{'type': 'text', 'text': "I don't think 'GOOGLE' is a valid stock symbol. Did you mean GOOGL or GOOG? Also, how many stocks would you like to buy?", 'extras': {'signature': 'CscFAQw51scA3NH+HndO+JMDiLUprXezPHQ7BjkE+jD3ovHmjcUa63NJ19LY9PgSFan7maiVjTgKfmzp2jjkHef51zVy0jfROL6VlcmCR20S+K/RmP5LS+xav0n9prVyPKkqk2aHSWoa+M9Piy0uL1+Jr+ddGJeLAZU1dDO41HFJcBcMjuACau7b0oLdqD6IQ27cRnWsraOSDqfJF8mc2H9HS13q23m4BDro/VYv/MmIXpj7NSLWPLhmMb0oS5l5getFbgj/FBfuLavkZwqRsQ8MBWAXQzp0OacCJtakqTtKctpLlr6RmfKtYPkjEPByvp/s91mgQav/f1GyfiUxr5Vbd7D18hksZskiNUyg2qzn1U7Y3z10Vol/L0VFF8IYJQu3T3dpDIJgbZQUNrs3K4PgjQ8n3yRnzFufR6nlr5u3QVg9Vto/qW2zMfHYapTeIH1H1j6X+DzfJeH8c9NhG12K/pPwUwYj1Z8u5paSAIqrx/eFc+VZpG8/J12UanvIfdXBuGoOJgef8KKWfja9o3xW+m2iqxbnJr3vQnmiekie8lIqAPXX0FTzqE909te1kx9eZeft+dvUY3Yz35Q1aqdGSY50/HpLfVkXbFLp0xavST56mIViGzligxFTBsS35dAnfMLN4FrqBjF8HWAY8yD502OPOBOdyewa8VHboCnCNo1hbiciuks5oe15PkoGfP/S4e8NPUyL0KCKvjvH7EX0dXt1Mi2+n13cpK7hplfXk1kfzI9elwB18vaKZKxfUNIBDr+Q5wsQdLOFMlt3hGzWopCgVhKdUpu42SSSkSlO5ZD/WGKT8Mn31